# Credit Card Fraud Detection — Model Training

This notebook covers the full machine-learning pipeline for training a **Logistic Regression** classifier to detect fraudulent credit card transactions using the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

## Outline
1. Load and inspect the dataset
2. Exploratory Data Analysis (EDA)
3. Preprocessing and feature engineering
4. Model training with PySpark MLlib
5. Model evaluation
6. Save the trained model for real-time inference

In [ ]:
# Install required packages (if running outside the virtual environment)
# !pip install pyspark pandas matplotlib seaborn

## 1. Initialise Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FraudDetectionModelTraining")
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Load the Dataset

Download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) and place it in the `data/` directory.

In [ ]:
DATA_PATH = "../data/creditcard.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(DATA_PATH)
)

print(f"Dataset shape: {df.count()} rows × {len(df.columns)} columns")
df.printSchema()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert to Pandas for visualisation (sample if dataset is large)
pdf = df.toPandas()

# Class distribution
class_counts = pdf["Class"].value_counts()
print("Class distribution:")
print(class_counts)
print(f"\nFraud rate: {class_counts[1] / len(pdf) * 100:.4f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(
    class_counts.values,
    labels=["Legitimate", "Fraud"],
    autopct="%1.2f%%",
    colors=["steelblue", "salmon"],
    startangle=90,
)
axes[0].set_title("Transaction Class Distribution")

# Transaction amount distribution by class
for cls, color, label in [(0, "steelblue", "Legitimate"), (1, "salmon", "Fraud")]:
    axes[1].hist(
        pdf[pdf["Class"] == cls]["Amount"],
        bins=50,
        alpha=0.6,
        color=color,
        label=label,
        density=True,
    )
axes[1].set_xlabel("Transaction Amount")
axes[1].set_ylabel("Density")
axes[1].set_title("Amount Distribution by Class")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (PCA features V1–V10)
plt.figure(figsize=(12, 8))
v_cols = [f"V{i}" for i in range(1, 11)]
corr = pdf[v_cols + ["Amount", "Class"]].corr()
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Feature columns: V1–V28 (PCA-transformed), Amount, Time
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Amount", "Time"]
label_col = "Class"

# Rename label column and cast to double (required by MLlib)
df_clean = (
    df
    .dropna(subset=feature_cols + [label_col])
    .withColumn("label", F.col(label_col).cast("double"))
)

print(f"Clean dataset: {df_clean.count()} rows")
df_clean.groupBy("label").count().show()

In [ ]:
# Train / test split (stratified by label)
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows : {train_df.count()}")
print(f"Test rows     : {test_df.count()}")

## 5. Build & Train the ML Pipeline

In [ ]:
# Pipeline stages
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler    = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withMean=True, withStd=True,
)
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
)

pipeline = Pipeline(stages=[assembler, scaler, lr])

# Hyperparameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,  [0.01, 0.1])
    .addGrid(lr.maxIter,   [50, 100])
    .build()
)

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
)

print("Training model (this may take a few minutes)...")
cv_model = cv.fit(train_df)
best_model = cv_model.bestModel
print("Training complete.")

## 6. Evaluation

In [ ]:
predictions = best_model.transform(test_df)

roc_auc = evaluator.evaluate(predictions)

mc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
f1        = mc_eval.evaluate(predictions, {mc_eval.metricName: "f1"})
precision = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedPrecision"})
recall    = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedRecall"})

print("=" * 40)
print("Model Evaluation (Test Set)")
print("=" * 40)
print(f"  AUC-ROC   : {roc_auc:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")

In [ ]:
# Confusion matrix using Pandas
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred_pdf = predictions.select("label", "prediction").toPandas()
cm = confusion_matrix(pred_pdf["label"], pred_pdf["prediction"])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Fraud"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Logistic Regression")
plt.tight_layout()
plt.show()

## 7. Save the Model

In [ ]:
MODEL_PATH = "../models/logistic_regression_fraud_model"

best_model.write().overwrite().save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

## 8. (Optional) Load and Verify Saved Model

In [ ]:
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(MODEL_PATH)
sample_preds = loaded_model.transform(test_df.limit(5))
sample_preds.select("label", "prediction", "probability").show(truncate=False)
print("Model loaded and verified successfully.")

---
## Summary

The trained Logistic Regression model is now saved and ready for use in the **real-time Spark Streaming pipeline** (`src/streaming/stream_processor.py`).  
The model file can be found at `models/logistic_regression_fraud_model`.